# Data Analytics Club - Python Workshop: Working with APIs

**Goal:** learn a repeatable way to retrieve live data from an API, inspect the response, turn it into a pandas DataFrame, and visualize it.

We will use two APIs:
1. **PokéAPI** - a simple, fun JSON API: https://pokeapi.co/docs/v2
2. **Open-Meteo** - a weather API with parameters and JSON/CSV output: https://open-meteo.com/en/docs

You do **not** need to have attended the previous workshop. We will move quickly through Python/pandas basics and focus on the API workflow.

### The pattern to remember
`endpoint -> request -> status code -> response -> JSON/CSV -> DataFrame -> analysis/plot`

**Classroom fallback:** `OFFLINE_MODE = True` is the default. It uses the supplied classroom files and sends no requests. To practice live GET requests, set it to `False` and restart/run from the top. Neither example requires a secret API key for this educational use. Offline weather is illustrative, not a forecast. In the student notebook, complete the fill-ins before running dependent cells.

## 0. Setup
Import the libraries we will use. `requests` handles web requests, `pandas` handles tabular data, and Plotly makes interactive charts.

In [ ]:
import json
from io import StringIO

import pandas as pd
import plotly.express as px
import requests

OFFLINE_MODE = True  # Set False to practice live requests; True requires no network.

from pathlib import Path

# Works from the repository root or this workshop folder.
WORKSHOP_DIR = Path.cwd()
if (WORKSHOP_DIR / "working-with-apis").is_dir():
    WORKSHOP_DIR = WORKSHOP_DIR / "working-with-apis"
DATA_DIR = WORKSHOP_DIR / "data"

# Reuse successful responses when rerunning cells in this kernel.
pokemon_cache = {}


## 1. Our first API request: PokéAPI
An **endpoint** is the URL where we ask an API for a resource. Here, the resource is one Pokémon.

We will request Pikachu, check the HTTP status code, then convert the response body from JSON into a Python dictionary.

In [ ]:
pokemon_name = "pikachu"
url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_name}"

if OFFLINE_MODE:
    with open(DATA_DIR / "pikachu_fallback.json", encoding="utf-8") as f:
        pokemon = json.load(f)
    print("Offline fallback loaded")
elif pokemon_name in pokemon_cache:
    pokemon = pokemon_cache[pokemon_name]
    print("Cached response loaded")
else:
    # TODO 1: send a GET request to url. Add timeout=10.
    response = ______________________________

    # TODO 2: print the response status code.
    print(______________________________)

    # TODO 3: raise an error if the request was not successful.
    ______________________________

    # TODO 4: convert the JSON response to a Python dictionary.
    pokemon = ______________________________

if not OFFLINE_MODE:
    pokemon_cache[pokemon_name] = pokemon

pokemon["name"]

### Inspecting JSON
JSON usually becomes nested Python **dictionaries** and **lists**. You do not memorize the structure - inspect it and follow the keys.

In [ ]:
print(type(pokemon))
print(pokemon.keys())

In [ ]:
# TODO 5: pull out a few values from the nested response.
name = pokemon[________________]
height_m = pokemon[________________] / 10
weight_kg = pokemon[________________] / 10
primary_type = pokemon["types"][0]["type"][________________]
attack = pokemon["stats"][1][________________]

print(name.title(), height_m, weight_kg, primary_type, attack)

## 2. From repeated API responses to a DataFrame
In real analysis, one response is rarely enough. We will request several Pokémon, extract the fields we care about, and turn the resulting rows into a DataFrame.

The helper function below is already written so we can focus on the API/data workflow rather than spending the whole meeting on loops and nested dictionaries.

In [ ]:
def pokemon_to_row(data):
    stats = {item["stat"]["name"]: item["base_stat"] for item in data["stats"]}
    types = [item["type"]["name"] for item in data["types"]]
    return {
        "name": data["name"].title(),
        "primary_type": types[0],
        "secondary_type": types[1] if len(types) > 1 else None,
        "height_m": data["height"] / 10,
        "weight_kg": data["weight"] / 10,
        "hp": stats["hp"],
        "attack": stats["attack"],
        "defense": stats["defense"],
        "sp_attack": stats["special-attack"],
        "sp_defense": stats["special-defense"],
        "speed": stats["speed"],
    }

In [ ]:
pokemon_names = [
    "bulbasaur", "charizard", "blastoise", "pikachu", "gengar", "dragonite",
    "lucario", "gardevoir", "snorlax", "eevee", "gyarados", "tyranitar"
]

if OFFLINE_MODE:
    with open(DATA_DIR / "pokemon_table_fallback.json", encoding="utf-8") as f:
        rows = json.load(f)
else:
    rows = []
    for name in pokemon_names:
        if name not in pokemon_cache:
            # TODO 6: request each Pokémon by inserting name into the endpoint.
            r = requests.get(f"https://pokeapi.co/api/v2/pokemon/{________________}", timeout=10)
            r.raise_for_status()
            pokemon_cache[name] = r.json()
        rows.append(pokemon_to_row(pokemon_cache[name]))

# TODO 7: turn rows into a pandas DataFrame.
pokemon_df = ______________________________
pokemon_df

### Quick analysis + visualization
This is intentionally brief: the last workshop focused on pandas/Plotly. The point here is that the data arrived from live API responses rather than a CSV we were handed.

Offline mode uses the fixed 12-row classroom table; it does not request new Pokémon.

In [ ]:
# TODO 8: sort by attack from highest to lowest and show the top 5.
pokemon_df.sort_values(________________, ascending=________________).head(5)

In [ ]:
# TODO 9: complete the Plotly scatter plot.
fig = px.scatter(
    pokemon_df,
    x=________________,
    y=________________,
    color="primary_type",
    size="hp",
    hover_name="name",
    title="Pokémon Attack vs. Defense"
)
fig.show()

## 3. A different API: Open-Meteo
Different APIs have different endpoints and response shapes, but the workflow is the same.

This time, instead of putting the resource directly in the URL, we send **query parameters**. We will ask for 3 days of hourly temperature and precipitation probability for Dayton.

Live weather data: [Open-Meteo](https://open-meteo.com/), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The lesson selects fields, renames columns, and parses time. The supplied offline values are illustrative classroom data, not Open-Meteo observations or forecasts.

In [ ]:
weather_url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 39.7589,
    "longitude": -84.1916,
    # TODO 10: request both temperature_2m and precipitation_probability.
    "hourly": ______________________________,
    "temperature_unit": "fahrenheit",
    "timezone": "America/New_York",
    "forecast_days": 3,
}

if OFFLINE_MODE:
    with open(DATA_DIR / "weather_offline_example.json", encoding="utf-8") as f:
        weather = json.load(f)
    print("Illustrative offline weather loaded")
else:
    # TODO 11: request weather_url and pass params=params.
    weather_response = ______________________________
    print(weather_response.status_code)
    weather_response.raise_for_status()
    weather = weather_response.json()

print(weather.keys())
print(weather["hourly"].keys())

In [ ]:
# TODO 12: build a DataFrame from the three parallel lists in weather["hourly"].
weather_df = pd.DataFrame({
    "time": weather["hourly"][________________],
    "temperature_f": weather["hourly"][________________],
    "precip_probability": weather["hourly"][________________],
})

# Convert the time strings into real datetimes so Plotly treats the x-axis as time.
weather_df["time"] = pd.to_datetime(weather_df["time"])
weather_df.head()

In [ ]:
# TODO 13: create a line chart of temperature over time.
fig = px.line(
    weather_df,
    x=________________,
    y=________________,
    title="Dayton: Illustrative Offline Weather" if OFFLINE_MODE else "Dayton: Hourly Temperature Forecast"
)
fig.show()

## 4. Same API, different response format: CSV
Open-Meteo can also return CSV. This is useful because it shows that the **API** and the **response format** are separate ideas.

For the live request we add `format=csv`. Open-Meteo places a few metadata lines before the hourly table, so `skiprows=3` jumps to the tabular portion.

In [ ]:
if OFFLINE_MODE:
    weather_csv_df = pd.read_csv(DATA_DIR / "weather_offline_example.csv")
else:
    csv_params = params.copy()
    # TODO 14: ask Open-Meteo for CSV instead of JSON.
    csv_params[________________] = ________________
    csv_response = requests.get(weather_url, params=csv_params, timeout=10)
    csv_response.raise_for_status()

    # requests gives us the CSV as text; StringIO lets pandas read that text like a file.
    weather_csv_df = pd.read_csv(StringIO(csv_response.text), skiprows=3)

weather_csv_df.head()

## 5. Mini challenge
Choose **one** change and try it without copying a completed answer:

- Change `pokemon_name` near the top and inspect a different Pokémon.
- Add/remove Pokémon in `pokemon_names` and rerun the DataFrame + chart.
- Change the Plotly axes (for example, `speed` vs. `attack`).
- Use one of the coordinates below and rerun the Open-Meteo request:
  - Cincinnati: `39.1031, -84.5120`
  - Columbus: `39.9612, -82.9988`
  - Nashville: `36.1627, -86.7816`
- Add `precipitation_probability` to a second Plotly chart.

### What you should remember
You will almost never memorize an API. Start with its documentation and ask:
1. What is the endpoint?
2. What parameters are available?
3. Does it require authentication?
4. What format comes back?
5. What does the returned data structure look like?

**Offline limits:** changing the Pokémon name/list or city coordinates does not change the fixed fallback files. Try chart-axis changes and the precipitation chart offline; use live mode for new resources or locations.

### Optional if we have time: what does a failed request look like?
Common status codes: **200** success, **400** bad request, **401/403** auth/permission, **404** not found, **429** too many requests.

In [ ]:
# Optional: try a resource that does not exist (live mode only).
if OFFLINE_MODE:
    print("404 demo skipped in offline mode; no request was sent.")
else:
    bad_response = requests.get("https://pokeapi.co/api/v2/pokemon/not-a-real-pokemon", timeout=10)
    print(bad_response.status_code)  # Expected: 404; the live service may vary.